# Tema 24 — Fusión multimodal: descripción de imágenes (BLIP) + clasificación de texto

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-12/Tema-24/Tema_24.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

En esta práctica construimos un **pipeline multimodal** que combina **visión** y **texto**:

1. Cargamos imágenes con sus descripciones (dataset **Flickr8k**).
2. Generamos una **descripción automática** de cada imagen con el modelo **BLIP** (*image captioning*).
3. **Fusionamos** el texto original con la descripción generada y entrenamos un **clasificador de texto** (TF-IDF + Regresión Logística) que predice una etiqueta de *prioridad* demostrativa.

> Nota: la etiqueta de prioridad es **didáctica**; su único fin es ilustrar el flujo completo *imagen → texto → clasificación*.

## 1. Instalación de dependencias

Este notebook usa `transformers` (para **BLIP**), `datasets`, `scikit-learn`, `torch`, `pandas` y `matplotlib`. En **Google Colab** casi todo viene preinstalado; la siguiente celda instala lo necesario **solo si estás en Colab**. En local, instala las dependencias una vez siguiendo el `README.md` de la carpeta.

In [ ]:
# === Instalación de dependencias (solo en Google Colab) ===
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q transformers datasets torch scikit-learn pandas matplotlib pillow
    print("Setup de Colab completado.")
else:
    print("Entorno local detectado. Asegúrate de tener: transformers, datasets, torch, scikit-learn, pandas, matplotlib (ver README.md).")

## 2. Importaciones

Importamos las librerías del pipeline: `pandas` para los datos, `datasets` para descargar **Flickr8k**, `transformers` para el modelo **BLIP**, y varias utilidades de `scikit-learn` para vectorizar el texto, entrenar el clasificador y evaluarlo.

In [ ]:
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

## 3. Cargar el dataset (Flickr8k)

Descargamos un subconjunto ligero (300 ejemplos) del dataset público **Flickr8k**, donde cada imagen trae **5 descripciones** escritas por personas. Lo convertimos en un `DataFrame` de `pandas` y usamos la **primera** descripción (`caption_0`) como *texto base* de cada imagen.

In [ ]:
# 1. Cargar un subconjunto ligero del dataset público Flickr8k
dataset = load_dataset("jxie/flickr8k", split="train[:300]")

# 2. Convertir a DataFrame
rows = []
for ex in dataset:
    captions = [ex["caption_0"], ex["caption_1"], ex["caption_2"], ex["caption_3"], ex["caption_4"]]
    rows.append({
        "image": ex["image"],
        "report_text": captions[0]  # usar el primer caption como texto base
    })

df = pd.DataFrame(rows)

## 4. Crear una etiqueta demostrativa

Para ilustrar un problema de **clasificación** definimos una etiqueta de *prioridad* (`alta`, `media`, `baja`) con reglas simples basadas en palabras clave del texto. **Importante:** esta etiqueta **no** representa una verdad de negocio real; solo sirve para mostrar el pipeline completo de principio a fin.

In [ ]:
# 3. Crear una etiqueta demostrativa para fines didácticos
# Esta etiqueta NO representa una verdad de negocio; solo sirve para mostrar el pipeline completo.
def prioridad_demo(texto: str) -> str:
    t = texto.lower()
    if any(k in t for k in ["dog", "dogs", "running", "jumping", "bike", "street"]):
        return "alta"
    elif any(k in t for k in ["child", "children", "park", "water", "grass"]):
        return "media"
    return "baja"

df["label"] = df["report_text"].apply(prioridad_demo)

## 5. Generar descripciones automáticas con BLIP

Usamos el modelo **BLIP** (`Salesforce/blip-image-captioning-base`) a través de un `pipeline` de *image-to-text* de Hugging Face. El código **detecta la GPU automáticamente**: usa **CUDA** (`device=0`) si hay una GPU disponible y, si no, cae a **CPU** (`device=-1`). Para mantener la práctica ligera trabajamos con una **muestra pequeña** (120 imágenes) y generamos una **descripción automática** de cada una.

> La primera ejecución descarga los pesos del modelo (unos cientos de MB) y puede tardar. En **CPU** el *captioning* de las 120 imágenes tarda algunos minutos; con **GPU** es mucho más rápido (en Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*).

In [ ]:
# 4. Modelo ligero para generar caption automático de la imagen
import torch

# Detección automática de GPU: usa CUDA (device=0) si está disponible; si no, CPU (device=-1).
device = 0 if torch.cuda.is_available() else -1
print("GPU (CUDA) detectada: usando GPU." if device == 0 else "Sin GPU: usando CPU.")

captioner = pipeline(
    task="image-to-text",
    model="Salesforce/blip-image-captioning-base",
    device=device   # GPU si está disponible, si no CPU
)

# Para que la práctica siga siendo ligera, trabajar con una muestra más pequeña
df = df.head(120).copy()

def generar_caption(img):
    salida = captioner(img)
    return salida[0]["generated_text"].strip()

df["image_caption"] = df["image"].apply(generar_caption)

## 6. Fusionar texto + descripción y separar en entrenamiento/prueba

**Fusionamos** en un solo campo el *texto original* de la imagen y la *descripción generada* por BLIP. Este texto combinado es la entrada del clasificador. Después separamos los datos en **entrenamiento (70%)** y **prueba (30%)** de forma **estratificada** para conservar la proporción de clases.

In [ ]:
# 5. Fusionar texto original + caption generado
df["fused_text"] = (
    "texto_original: " + df["report_text"].astype(str) +
    " | descripcion_imagen: " + df["image_caption"].astype(str)
)

# 6. Separar entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    df["fused_text"],
    df["label"],
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

## 7. Vectorizar (TF-IDF) y entrenar el clasificador

Convertimos el texto fusionado en vectores numéricos con **TF-IDF** (usando *unigramas* y *bigramas*) y entrenamos una **Regresión Logística** con `class_weight="balanced"` para compensar el desbalance entre clases.

In [ ]:
# 7. Vectorizar el texto fusionado
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 8. Entrenar clasificador
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(X_train_vec, y_train)

## 8. Predecir y evaluar

Generamos las predicciones sobre el conjunto de prueba y evaluamos el modelo con un **reporte de clasificación** (precisión, *recall*, F1) y una **matriz de confusión**. Al final mostramos algunos ejemplos con su etiqueta real y la predicha.

In [ ]:
# 9. Predecir
y_pred = clf.predict(X_test_vec)

# 10. Evaluar
print(classification_report(y_test, y_pred))

labels_sorted = sorted(df["label"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_sorted)
disp.plot(xticks_rotation=45)
plt.tight_layout()
plt.show()

# 11. Mostrar algunos resultados
resultados = pd.DataFrame({
    "texto_fusionado": X_test.values,
    "label_real": y_test.values,
    "label_pred": y_pred
})
print(resultados.head(10))

## 9. Evaluación del prototipo

Volvemos a mostrar el **reporte de clasificación** y la **matriz de confusión** de forma aislada, para analizar el desempeño del prototipo por clase.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Reporte de clasificación
print(classification_report(y_test, y_pred))

# Matriz de confusión
labels_sorted = sorted(df["label"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels_sorted
)
disp.plot(xticks_rotation=45)
plt.tight_layout()
plt.show()